# Water Usage Optimization for Cooling - Synthetic Data Generator

## Overview
This notebook generates realistic synthetic data for the **Water Usage Optimization for Cooling** use case in data center infrastructure. The data simulates water system telemetry from cooling towers, flow meters, water quality sensors, and related equipment.

## What It Simulates
- **Cooling tower operations** with realistic water consumption patterns
- **Water quality measurements** (conductivity, pH, TDS)
- **Flow meter readings** for makeup water, blowdown, and distribution
- **Weather data** affecting evaporation rates
- **Anomalies** including leaks, quality drift, and equipment issues

## Tables/Streams Created
| Table | Description |
|-------|-------------|
| `dim_cooling_towers` | Cooling tower equipment reference data |
| `dim_sensors` | Sensor inventory and specifications |
| `fact_flow_readings` | Flow meter measurements (gallons/minute) |
| `fact_water_quality` | Water quality sensor readings |
| `fact_cooling_tower_telemetry` | Cooling tower operational data |
| `fact_weather_readings` | Ambient weather conditions |
| `fact_blowdown_events` | Blowdown cycle events |
| `fact_leak_alerts` | Generated leak detection alerts |

## How to Tune Volume
Adjust the parameters in the **Configuration** section below:
- `SCALE_FACTOR`: Multiplier for data volume (1 = baseline, 2 = 2x data)
- `NUM_COOLING_TOWERS`: Number of cooling towers to simulate
- `READINGS_PER_MINUTE`: Sensor reading frequency
- `DATE_RANGE`: Historical backfill period

## Known Limitations
- Synthetic data uses statistical distributions; real systems may have site-specific patterns
- Leak events are randomly injected; actual leak signatures vary by equipment
- Chemical treatment is simplified; real systems have complex dosing schedules

---

## How to Run in Microsoft Fabric

### Prerequisites
1. Create a **Lakehouse** in your Fabric workspace
2. Attach this notebook to the Lakehouse
3. Ensure you have write permissions to the Lakehouse

### Running the Notebook
1. **Batch Mode (Historical Backfill):**
   - Set `MODE = 'batch'` in the Configuration section
   - Run all cells to generate historical data
   - Data will be written as Delta tables to your Lakehouse

2. **Streaming Simulation Mode:**
   - Set `MODE = 'streaming'` in the Configuration section
   - Schedule the notebook to run every 1-5 minutes
   - Each run appends new events simulating real-time data

### Recommended Schedule Settings
- **For demos:** Run every 1 minute for realistic real-time experience
- **For testing:** Run every 5 minutes to reduce compute costs

### Idempotent Behavior
- **Batch mode:** Uses `overwrite` mode - safe to rerun
- **Streaming mode:** Uses `append` mode - each run adds new records

---

In [ ]:
# Install required packages (run once if needed)
!pip install faker

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Mode: 'batch' for historical backfill, 'streaming' for incremental events
MODE = 'batch'

# Scale factor (1 = baseline, increase for more data)
SCALE_FACTOR = 1

# Random seed for reproducibility
RANDOM_SEED = 42

# Date range for batch mode
START_DATE = '2025-01-01'
END_DATE = '2025-03-01'

# Equipment configuration
NUM_COOLING_TOWERS = 8
NUM_BUILDINGS = 4

# Sensor reading frequency (readings per hour per sensor)
READINGS_PER_HOUR = 60  # 1 per minute

# Streaming mode: events per run
STREAMING_EVENTS_PER_RUN = 100

# Anomaly injection rate (probability)
LEAK_PROBABILITY = 0.002
QUALITY_ANOMALY_PROBABILITY = 0.005

# Output configuration
OUTPUT_FORMAT = 'delta'  # 'delta' or 'csv'
LAKEHOUSE_PATH = 'Tables'  # Path in Lakehouse for Delta tables

print(f"Configuration loaded:")
print(f"  Mode: {MODE}")
print(f"  Scale Factor: {SCALE_FACTOR}")
print(f"  Date Range: {START_DATE} to {END_DATE}")
print(f"  Cooling Towers: {NUM_COOLING_TOWERS}")

In [ ]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
import uuid
import warnings
warnings.filterwarnings('ignore')

# Initialize random generators with seed
np.random.seed(RANDOM_SEED)
fake = Faker()
Faker.seed(RANDOM_SEED)

print("Libraries imported successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## Dimension Tables
Generate reference data for cooling towers and sensors.

In [ ]:
# =============================================================================
# DIMENSION: COOLING TOWERS
# =============================================================================

def generate_cooling_towers(num_towers, num_buildings):
    """Generate cooling tower dimension table."""
    
    towers = []
    for i in range(num_towers):
        tower_id = f"CT-{str(i+1).zfill(3)}"
        building_id = f"BLD-{chr(65 + (i % num_buildings))}"
        
        towers.append({
            'tower_id': tower_id,
            'tower_name': f"Cooling Tower {i+1}",
            'building_id': building_id,
            'capacity_tons': np.random.choice([500, 750, 1000, 1500]),
            'design_flow_gpm': np.random.choice([1500, 2250, 3000, 4500]),
            'cycles_of_concentration_target': np.random.uniform(4.0, 6.0),
            'manufacturer': np.random.choice(['Evapco', 'BAC', 'Marley', 'SPX']),
            'install_date': fake.date_between(start_date='-10y', end_date='-1y'),
            'last_maintenance_date': fake.date_between(start_date='-6m', end_date='today'),
            'latitude': 37.3861 + np.random.uniform(-0.01, 0.01),
            'longitude': -122.0839 + np.random.uniform(-0.01, 0.01)
        })
    
    return pd.DataFrame(towers)

dim_cooling_towers = generate_cooling_towers(NUM_COOLING_TOWERS, NUM_BUILDINGS)
print(f"Generated {len(dim_cooling_towers)} cooling towers")
dim_cooling_towers.head()

In [ ]:
# =============================================================================
# DIMENSION: SENSORS
# =============================================================================

def generate_sensors(towers_df):
    """Generate sensor dimension table."""
    
    sensor_types = [
        {'type': 'flow_meter', 'unit': 'gpm', 'locations': ['makeup', 'blowdown', 'distribution']},
        {'type': 'conductivity', 'unit': 'µS/cm', 'locations': ['basin', 'makeup']},
        {'type': 'ph', 'unit': 'pH', 'locations': ['basin']},
        {'type': 'temperature', 'unit': '°F', 'locations': ['supply', 'return', 'basin']},
        {'type': 'level', 'unit': 'inches', 'locations': ['basin']}
    ]
    
    sensors = []
    sensor_counter = 1
    
    for _, tower in towers_df.iterrows():
        for sensor_type in sensor_types:
            for location in sensor_type['locations']:
                sensors.append({
                    'sensor_id': f"SNS-{str(sensor_counter).zfill(4)}",
                    'tower_id': tower['tower_id'],
                    'sensor_type': sensor_type['type'],
                    'location': location,
                    'unit': sensor_type['unit'],
                    'manufacturer': np.random.choice(['Siemens', 'Honeywell', 'Endress+Hauser', 'Yokogawa']),
                    'calibration_date': fake.date_between(start_date='-1y', end_date='today'),
                    'is_active': True
                })
                sensor_counter += 1
    
    return pd.DataFrame(sensors)

dim_sensors = generate_sensors(dim_cooling_towers)
print(f"Generated {len(dim_sensors)} sensors")
dim_sensors.head(10)

## Fact Tables
Generate time-series data for flow readings, water quality, and operational telemetry.

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_timestamps(start_date, end_date, readings_per_hour):
    """Generate timestamp series for readings."""
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    freq_minutes = 60 // readings_per_hour
    return pd.date_range(start=start, end=end, freq=f'{freq_minutes}min')

def add_seasonality(base_value, timestamp, daily_amplitude=0.1, weekly_amplitude=0.05):
    """Add realistic daily and weekly seasonality to a value."""
    hour = timestamp.hour
    day_of_week = timestamp.dayofweek
    
    # Daily pattern: higher during business hours
    daily_factor = 1 + daily_amplitude * np.sin(2 * np.pi * (hour - 6) / 24)
    
    # Weekly pattern: slightly lower on weekends
    weekly_factor = 1 - weekly_amplitude * (day_of_week >= 5)
    
    return base_value * daily_factor * weekly_factor

def add_temperature_correlation(base_value, ambient_temp, sensitivity=0.02):
    """Correlate value with ambient temperature (higher temp = higher value)."""
    temp_factor = 1 + sensitivity * (ambient_temp - 70)  # 70°F as baseline
    return base_value * temp_factor

print("Helper functions defined.")

In [ ]:
# =============================================================================
# FACT: WEATHER READINGS
# =============================================================================

def generate_weather_data(start_date, end_date):
    """Generate hourly weather data."""
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq='1H')
    
    weather_data = []
    for ts in timestamps:
        month = ts.month
        hour = ts.hour
        
        # Seasonal base temperature (Northern California climate)
        seasonal_base = 55 + 15 * np.sin(2 * np.pi * (month - 4) / 12)
        
        # Daily variation
        daily_variation = 10 * np.sin(2 * np.pi * (hour - 6) / 24)
        
        ambient_temp = seasonal_base + daily_variation + np.random.normal(0, 3)
        
        # Humidity inversely related to temperature
        humidity = max(20, min(95, 70 - 0.5 * (ambient_temp - 60) + np.random.normal(0, 10)))
        
        # Wet bulb temperature calculation (simplified)
        wet_bulb_temp = ambient_temp - (100 - humidity) / 5
        
        weather_data.append({
            'timestamp': ts,
            'ambient_temp_f': round(ambient_temp, 1),
            'relative_humidity_pct': round(humidity, 1),
            'wet_bulb_temp_f': round(wet_bulb_temp, 1),
            'wind_speed_mph': max(0, np.random.exponential(5)),
            'precipitation_in': np.random.choice([0, 0, 0, 0, 0, 0.01, 0.05, 0.1], p=[0.85, 0.05, 0.03, 0.02, 0.02, 0.01, 0.01, 0.01])
        })
    
    return pd.DataFrame(weather_data)

fact_weather_readings = generate_weather_data(START_DATE, END_DATE)
print(f"Generated {len(fact_weather_readings)} weather readings")
fact_weather_readings.head()

In [ ]:
# =============================================================================
# FACT: FLOW READINGS
# =============================================================================

def generate_flow_readings(towers_df, sensors_df, weather_df, start_date, end_date, scale_factor=1):
    """Generate flow meter readings."""
    
    # Get flow sensors
    flow_sensors = sensors_df[sensors_df['sensor_type'] == 'flow_meter'].copy()
    
    # Generate timestamps (every 5 minutes for efficiency)
    timestamps = pd.date_range(start=start_date, end=end_date, freq='5min')
    
    # Sample timestamps based on scale factor
    if scale_factor < 1:
        timestamps = timestamps[::int(1/scale_factor)]
    
    readings = []
    leak_periods = {}  # Track active leaks per tower
    
    for ts in timestamps:
        # Get weather for this hour
        hour_ts = ts.floor('H')
        weather_row = weather_df[weather_df['timestamp'] == hour_ts]
        if len(weather_row) == 0:
            ambient_temp = 70
        else:
            ambient_temp = weather_row.iloc[0]['ambient_temp_f']
        
        for _, sensor in flow_sensors.iterrows():
            tower_id = sensor['tower_id']
            tower = towers_df[towers_df['tower_id'] == tower_id].iloc[0]
            
            # Base flow rates by location
            if sensor['location'] == 'makeup':
                # Makeup water - correlated with temperature
                base_flow = tower['design_flow_gpm'] * 0.03  # ~3% of circulation
                base_flow = add_temperature_correlation(base_flow, ambient_temp, 0.03)
            elif sensor['location'] == 'blowdown':
                # Blowdown - intermittent
                base_flow = tower['design_flow_gpm'] * 0.005  # ~0.5% when active
                if np.random.random() > 0.3:  # 70% of the time blowdown is minimal
                    base_flow = base_flow * 0.1
            else:  # distribution
                base_flow = tower['design_flow_gpm'] * np.random.uniform(0.6, 0.9)
            
            # Add seasonality
            flow = add_seasonality(base_flow, ts, daily_amplitude=0.15)
            
            # Check for leak injection
            if tower_id not in leak_periods:
                leak_periods[tower_id] = {'active': False, 'start': None, 'magnitude': 0}
            
            leak_info = leak_periods[tower_id]
            
            # Start new leak randomly
            if not leak_info['active'] and np.random.random() < LEAK_PROBABILITY:
                leak_info['active'] = True
                leak_info['start'] = ts
                leak_info['magnitude'] = np.random.uniform(1.1, 1.5)  # 10-50% excess
            
            # End leak after 2-8 hours
            if leak_info['active'] and (ts - leak_info['start']).total_seconds() > np.random.uniform(2, 8) * 3600:
                leak_info['active'] = False
            
            # Apply leak if active and this is makeup water
            is_leak = False
            if leak_info['active'] and sensor['location'] == 'makeup':
                flow = flow * leak_info['magnitude']
                is_leak = True
            
            # Add noise
            flow = max(0, flow + np.random.normal(0, flow * 0.02))
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'timestamp': ts,
                'sensor_id': sensor['sensor_id'],
                'tower_id': tower_id,
                'location': sensor['location'],
                'flow_gpm': round(flow, 2),
                'is_anomaly': is_leak
            })
    
    return pd.DataFrame(readings)

fact_flow_readings = generate_flow_readings(
    dim_cooling_towers, dim_sensors, fact_weather_readings,
    START_DATE, END_DATE, SCALE_FACTOR
)
print(f"Generated {len(fact_flow_readings)} flow readings")
print(f"Anomalies (leaks): {fact_flow_readings['is_anomaly'].sum()}")
fact_flow_readings.head()

In [ ]:
# =============================================================================
# FACT: WATER QUALITY READINGS
# =============================================================================

def generate_water_quality(towers_df, sensors_df, start_date, end_date, scale_factor=1):
    """Generate water quality sensor readings."""
    
    # Get quality sensors (conductivity, pH)
    quality_sensors = sensors_df[sensors_df['sensor_type'].isin(['conductivity', 'ph'])].copy()
    
    # Generate timestamps (every 15 minutes)
    timestamps = pd.date_range(start=start_date, end=end_date, freq='15min')
    
    readings = []
    
    for ts in timestamps:
        for _, sensor in quality_sensors.iterrows():
            tower_id = sensor['tower_id']
            tower = towers_df[towers_df['tower_id'] == tower_id].iloc[0]
            
            is_anomaly = False
            
            if sensor['sensor_type'] == 'conductivity':
                if sensor['location'] == 'basin':
                    # Basin conductivity should be cycles_of_concentration * makeup
                    target_cycles = tower['cycles_of_concentration_target']
                    makeup_conductivity = 400  # typical municipal water
                    base_value = makeup_conductivity * target_cycles
                    
                    # Add drift over time (simulating concentration buildup)
                    hour_of_day = ts.hour
                    drift = 50 * np.sin(2 * np.pi * hour_of_day / 24)
                    
                    value = base_value + drift + np.random.normal(0, 50)
                    
                    # Quality anomaly injection
                    if np.random.random() < QUALITY_ANOMALY_PROBABILITY:
                        value = value * np.random.uniform(1.2, 1.5)
                        is_anomaly = True
                else:  # makeup
                    value = 400 + np.random.normal(0, 20)
                
                readings.append({
                    'reading_id': str(uuid.uuid4()),
                    'timestamp': ts,
                    'sensor_id': sensor['sensor_id'],
                    'tower_id': tower_id,
                    'measurement_type': 'conductivity',
                    'value': round(max(100, value), 1),
                    'unit': 'µS/cm',
                    'is_anomaly': is_anomaly
                })
                
            elif sensor['sensor_type'] == 'ph':
                # pH should be controlled between 7.0 and 8.5
                base_ph = 7.8
                value = base_ph + np.random.normal(0, 0.2)
                
                # Quality anomaly injection
                if np.random.random() < QUALITY_ANOMALY_PROBABILITY:
                    value = value + np.random.choice([-0.8, 0.8])
                    is_anomaly = True
                
                readings.append({
                    'reading_id': str(uuid.uuid4()),
                    'timestamp': ts,
                    'sensor_id': sensor['sensor_id'],
                    'tower_id': tower_id,
                    'measurement_type': 'ph',
                    'value': round(max(6.0, min(9.0, value)), 2),
                    'unit': 'pH',
                    'is_anomaly': is_anomaly
                })
    
    return pd.DataFrame(readings)

fact_water_quality = generate_water_quality(
    dim_cooling_towers, dim_sensors, START_DATE, END_DATE, SCALE_FACTOR
)
print(f"Generated {len(fact_water_quality)} water quality readings")
print(f"Quality anomalies: {fact_water_quality['is_anomaly'].sum()}")
fact_water_quality.head()

In [ ]:
# =============================================================================
# FACT: COOLING TOWER TELEMETRY
# =============================================================================

def generate_tower_telemetry(towers_df, weather_df, start_date, end_date):
    """Generate cooling tower operational telemetry."""
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq='5min')
    
    telemetry = []
    
    for ts in timestamps:
        # Get weather for this hour
        hour_ts = ts.floor('H')
        weather_row = weather_df[weather_df['timestamp'] == hour_ts]
        if len(weather_row) == 0:
            ambient_temp = 70
            wet_bulb = 60
        else:
            ambient_temp = weather_row.iloc[0]['ambient_temp_f']
            wet_bulb = weather_row.iloc[0]['wet_bulb_temp_f']
        
        for _, tower in towers_df.iterrows():
            # Calculate load factor based on temperature
            load_factor = min(1.0, max(0.3, 0.5 + (ambient_temp - 60) * 0.015))
            load_factor = add_seasonality(load_factor, ts, daily_amplitude=0.1)
            
            # Fan speed correlates with load
            fan_speed_pct = load_factor * 100 * np.random.uniform(0.9, 1.1)
            fan_speed_pct = max(20, min(100, fan_speed_pct))
            
            # Basin level
            basin_level_in = 18 + np.random.normal(0, 0.5)  # Target 18 inches
            
            # Supply and return temperatures
            supply_temp = wet_bulb + 8 + np.random.normal(0, 0.5)  # Approach temp ~8°F
            return_temp = supply_temp + 10 * load_factor + np.random.normal(0, 0.5)
            
            # Calculate WUE (simplified)
            it_load_kw = tower['capacity_tons'] * 3.517 * load_factor  # kW
            water_usage_gal = tower['design_flow_gpm'] * 0.03 * 5  # 5 min of makeup
            wue = (water_usage_gal * 3.785) / it_load_kw if it_load_kw > 0 else 0  # L/kWh
            
            telemetry.append({
                'telemetry_id': str(uuid.uuid4()),
                'timestamp': ts,
                'tower_id': tower['tower_id'],
                'fan_speed_pct': round(fan_speed_pct, 1),
                'basin_level_in': round(basin_level_in, 2),
                'supply_temp_f': round(supply_temp, 1),
                'return_temp_f': round(return_temp, 1),
                'approach_temp_f': round(supply_temp - wet_bulb, 1),
                'load_factor': round(load_factor, 3),
                'calculated_wue': round(wue, 3)
            })
    
    return pd.DataFrame(telemetry)

fact_cooling_tower_telemetry = generate_tower_telemetry(
    dim_cooling_towers, fact_weather_readings, START_DATE, END_DATE
)
print(f"Generated {len(fact_cooling_tower_telemetry)} telemetry records")
fact_cooling_tower_telemetry.head()

In [ ]:
# =============================================================================
# FACT: BLOWDOWN EVENTS
# =============================================================================

def generate_blowdown_events(towers_df, quality_df):
    """Generate blowdown cycle events based on conductivity readings."""
    
    events = []
    
    # Get basin conductivity readings
    basin_conductivity = quality_df[
        (quality_df['measurement_type'] == 'conductivity') & 
        (quality_df['value'] > 2200)  # Threshold for blowdown
    ].copy()
    
    for _, tower in towers_df.iterrows():
        tower_readings = basin_conductivity[basin_conductivity['tower_id'] == tower['tower_id']]
        
        for _, reading in tower_readings.iterrows():
            # Create blowdown event
            duration_min = np.random.uniform(5, 20)
            volume_gal = tower['design_flow_gpm'] * 0.01 * duration_min
            
            events.append({
                'event_id': str(uuid.uuid4()),
                'timestamp': reading['timestamp'],
                'tower_id': tower['tower_id'],
                'trigger_conductivity': reading['value'],
                'duration_minutes': round(duration_min, 1),
                'volume_gallons': round(volume_gal, 1),
                'end_conductivity': round(reading['value'] * 0.7 + np.random.normal(0, 30), 1)
            })
    
    return pd.DataFrame(events)

fact_blowdown_events = generate_blowdown_events(dim_cooling_towers, fact_water_quality)
print(f"Generated {len(fact_blowdown_events)} blowdown events")
fact_blowdown_events.head()

In [ ]:
# =============================================================================
# FACT: LEAK ALERTS
# =============================================================================

def generate_leak_alerts(flow_df):
    """Generate leak detection alerts based on flow anomalies."""
    
    # Get flow readings with anomalies
    anomalies = flow_df[flow_df['is_anomaly'] == True].copy()
    
    if len(anomalies) == 0:
        return pd.DataFrame()
    
    # Group consecutive anomalies into alert events
    alerts = []
    current_alert = None
    
    for _, row in anomalies.sort_values(['tower_id', 'timestamp']).iterrows():
        if current_alert is None or current_alert['tower_id'] != row['tower_id']:
            if current_alert is not None:
                alerts.append(current_alert)
            current_alert = {
                'alert_id': str(uuid.uuid4()),
                'tower_id': row['tower_id'],
                'alert_start': row['timestamp'],
                'alert_end': row['timestamp'],
                'max_flow_excess_pct': 0,
                'severity': 'Medium',
                'status': 'Detected'
            }
        else:
            current_alert['alert_end'] = row['timestamp']
    
    if current_alert is not None:
        alerts.append(current_alert)
    
    alerts_df = pd.DataFrame(alerts)
    
    # Add calculated fields
    if len(alerts_df) > 0:
        alerts_df['duration_minutes'] = (alerts_df['alert_end'] - alerts_df['alert_start']).dt.total_seconds() / 60
        alerts_df['max_flow_excess_pct'] = np.random.uniform(10, 50, len(alerts_df))
        alerts_df['severity'] = alerts_df['duration_minutes'].apply(
            lambda x: 'High' if x > 60 else ('Medium' if x > 15 else 'Low')
        )
    
    return alerts_df

fact_leak_alerts = generate_leak_alerts(fact_flow_readings)
print(f"Generated {len(fact_leak_alerts)} leak alerts")
if len(fact_leak_alerts) > 0:
    display(fact_leak_alerts.head())

## Quick Validation
Verify data integrity and display summary statistics.

In [ ]:
# =============================================================================
# DATA VALIDATION
# =============================================================================

print("=" * 60)
print("DATA VALIDATION SUMMARY")
print("=" * 60)

# Row counts
print("\n📊 ROW COUNTS:")
tables = {
    'dim_cooling_towers': dim_cooling_towers,
    'dim_sensors': dim_sensors,
    'fact_flow_readings': fact_flow_readings,
    'fact_water_quality': fact_water_quality,
    'fact_cooling_tower_telemetry': fact_cooling_tower_telemetry,
    'fact_weather_readings': fact_weather_readings,
    'fact_blowdown_events': fact_blowdown_events,
    'fact_leak_alerts': fact_leak_alerts
}

for name, df in tables.items():
    print(f"  {name}: {len(df):,} rows")

# Null checks
print("\n🔍 NULL VALUE CHECKS:")
for name, df in tables.items():
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(f"  {name}: {null_counts.sum()} nulls")
    else:
        print(f"  {name}: ✓ No nulls")

# Key integrity
print("\n🔗 KEY INTEGRITY:")
tower_ids_dim = set(dim_cooling_towers['tower_id'])
tower_ids_flow = set(fact_flow_readings['tower_id'])
if tower_ids_flow.issubset(tower_ids_dim):
    print("  Flow readings → Cooling towers: ✓ Valid")
else:
    print("  Flow readings → Cooling towers: ✗ Invalid references")

sensor_ids_dim = set(dim_sensors['sensor_id'])
sensor_ids_flow = set(fact_flow_readings['sensor_id'])
if sensor_ids_flow.issubset(sensor_ids_dim):
    print("  Flow readings → Sensors: ✓ Valid")
else:
    print("  Flow readings → Sensors: ✗ Invalid references")

print("\n" + "=" * 60)

In [ ]:
# =============================================================================
# KPI TRENDS VISUALIZATION
# =============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Daily water consumption trend
daily_flow = fact_flow_readings[fact_flow_readings['location'] == 'makeup'].groupby(
    fact_flow_readings['timestamp'].dt.date
)['flow_gpm'].sum()

axes[0, 0].plot(daily_flow.index, daily_flow.values, color='#1864ab')
axes[0, 0].set_title('Daily Makeup Water Flow (GPM Total)', fontsize=12)
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Total GPM')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Average basin conductivity
basin_cond = fact_water_quality[fact_water_quality['measurement_type'] == 'conductivity'].groupby(
    fact_water_quality['timestamp'].dt.date
)['value'].mean()

axes[0, 1].plot(basin_cond.index, basin_cond.values, color='#e67700')
axes[0, 1].axhline(y=2200, color='red', linestyle='--', label='Blowdown Threshold')
axes[0, 1].set_title('Average Basin Conductivity (µS/cm)', fontsize=12)
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Conductivity')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. WUE trend
daily_wue = fact_cooling_tower_telemetry.groupby(
    fact_cooling_tower_telemetry['timestamp'].dt.date
)['calculated_wue'].mean()

axes[1, 0].plot(daily_wue.index, daily_wue.values, color='#2f9e44')
axes[1, 0].axhline(y=1.5, color='green', linestyle='--', label='Target WUE')
axes[1, 0].set_title('Average Water Usage Effectiveness (L/kWh)', fontsize=12)
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('WUE')
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Temperature vs Load correlation
temp_load = fact_cooling_tower_telemetry.merge(
    fact_weather_readings[['timestamp', 'ambient_temp_f']],
    left_on=fact_cooling_tower_telemetry['timestamp'].dt.floor('H'),
    right_on='timestamp',
    how='left'
).dropna()

axes[1, 1].scatter(temp_load['ambient_temp_f'], temp_load['load_factor'], 
                   alpha=0.1, s=1, color='#7048e8')
axes[1, 1].set_title('Ambient Temperature vs Load Factor', fontsize=12)
axes[1, 1].set_xlabel('Ambient Temperature (°F)')
axes[1, 1].set_ylabel('Load Factor')

plt.tight_layout()
plt.show()

print("\n📈 KPI Summary:")
print(f"  Average WUE: {daily_wue.mean():.2f} L/kWh")
print(f"  Peak daily makeup flow: {daily_flow.max():,.0f} GPM")
print(f"  Blowdown events: {len(fact_blowdown_events)}")
print(f"  Leak alerts: {len(fact_leak_alerts)}")

## Save Data to Lakehouse
Write all tables as Delta format to the attached Lakehouse.

In [ ]:
# =============================================================================
# SAVE TO LAKEHOUSE
# =============================================================================

def save_to_lakehouse(dataframes_dict, output_format='delta', mode='overwrite'):
    """Save dataframes to Lakehouse as Delta tables or CSV."""
    
    # Check if running in Fabric
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        in_fabric = True
        print("✓ Running in Microsoft Fabric environment")
    except:
        in_fabric = False
        print("⚠ Not running in Fabric - saving to local CSV files")
    
    saved_tables = []
    
    for table_name, df in dataframes_dict.items():
        if len(df) == 0:
            print(f"  Skipping {table_name} (empty)")
            continue
            
        if in_fabric and output_format == 'delta':
            # Convert to Spark DataFrame and save as Delta
            spark_df = spark.createDataFrame(df)
            spark_df.write.format("delta").mode(mode).saveAsTable(table_name)
            print(f"  ✓ Saved {table_name} as Delta table ({len(df):,} rows)")
        else:
            # Save as CSV locally
            filename = f"{table_name}.csv"
            df.to_csv(filename, index=False)
            print(f"  ✓ Saved {filename} ({len(df):,} rows)")
        
        saved_tables.append(table_name)
    
    return saved_tables

# Prepare all tables
all_tables = {
    'dim_cooling_towers': dim_cooling_towers,
    'dim_sensors': dim_sensors,
    'fact_flow_readings': fact_flow_readings,
    'fact_water_quality': fact_water_quality,
    'fact_cooling_tower_telemetry': fact_cooling_tower_telemetry,
    'fact_weather_readings': fact_weather_readings,
    'fact_blowdown_events': fact_blowdown_events,
    'fact_leak_alerts': fact_leak_alerts
}

print("\n" + "=" * 60)
print("SAVING DATA TO LAKEHOUSE")
print("=" * 60 + "\n")

write_mode = 'overwrite' if MODE == 'batch' else 'append'
saved = save_to_lakehouse(all_tables, OUTPUT_FORMAT, write_mode)

print(f"\n✓ Successfully saved {len(saved)} tables")

## Streaming Simulation Mode (Optional)
Run this section when `MODE = 'streaming'` to generate incremental events.

In [ ]:
# =============================================================================
# STREAMING SIMULATION
# =============================================================================

if MODE == 'streaming':
    from datetime import datetime
    
    current_time = datetime.now()
    
    print(f"\n🔄 Streaming mode - generating events for {current_time}")
    
    # Generate a small batch of new events
    streaming_start = current_time - timedelta(minutes=5)
    streaming_end = current_time
    
    # Generate weather for current period
    streaming_weather = generate_weather_data(
        streaming_start.strftime('%Y-%m-%d %H:%M'),
        streaming_end.strftime('%Y-%m-%d %H:%M')
    )
    
    # Generate flow readings
    streaming_flow = generate_flow_readings(
        dim_cooling_towers, dim_sensors, streaming_weather,
        streaming_start.strftime('%Y-%m-%d %H:%M'),
        streaming_end.strftime('%Y-%m-%d %H:%M'),
        scale_factor=1
    )
    
    # Generate water quality
    streaming_quality = generate_water_quality(
        dim_cooling_towers, dim_sensors,
        streaming_start.strftime('%Y-%m-%d %H:%M'),
        streaming_end.strftime('%Y-%m-%d %H:%M')
    )
    
    print(f"  Generated {len(streaming_flow)} flow events")
    print(f"  Generated {len(streaming_quality)} quality events")
    
    # Append to existing tables
    streaming_tables = {
        'fact_flow_readings': streaming_flow,
        'fact_water_quality': streaming_quality,
        'fact_weather_readings': streaming_weather
    }
    
    save_to_lakehouse(streaming_tables, OUTPUT_FORMAT, mode='append')
    print("\n✓ Streaming events appended")
else:
    print("\nℹ Streaming mode not active. Set MODE = 'streaming' to enable.")

---

## Summary

This notebook has generated synthetic data for the **Water Usage Optimization for Cooling** use case, including:

- ✅ Dimension tables for cooling towers and sensors
- ✅ Fact tables for flow readings, water quality, and operational telemetry
- ✅ Blowdown events and leak alerts with realistic anomaly injection
- ✅ Weather data correlated with cooling load

### Next Steps
1. **Eventstream**: Create an Eventstream to ingest the fact tables for real-time processing
2. **Eventhouse**: Load data into a KQL database for analysis
3. **Real-Time Dashboard**: Build dashboards using the generated data
4. **Activator**: Set up alerts based on leak detection and quality thresholds

---
*Generated: March 2026 | Use Case Reference: DC-008*